### Predicting Freight Cost

In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# Establishing connection with inventory 
conn = sqlite3.connect("C:/Users/Aadarsh/Desktop/Vendor Invoice Intelligent System/data/inventory.db")

In [ ]:
query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""

tables = pd.read_sql_query(query, conn)

In [ ]:
tables

In [ ]:
for table in tables["name"]:
    print(f"\nTable: {table}")

    df = pd.read_sql_query(
        f"SELECT * FROM {table} LIMIT 5",
        conn
    )
    display(df)

In [ ]:
vendor_df = pd.read_sql_query("select * from vendor_invoice", conn)

In [ ]:
vendor_df.head()

In [ ]:
vendor_df[["Quantity", "Freight", "Dollars"]].corr()

In [ ]:
# Relationship between Quantity, Dollars and Freight
plt.figure(figsize=(4,2))
sns.heatmap(vendor_df[["Quantity", "Freight", "Dollars"]].corr(), annot=True)
plt.show()

plt.scatter(vendor_df["Quantity"], vendor_df["Freight"], color="#f57a55")
plt.scatter(vendor_df["Dollars"], vendor_df["Freight"], color="#7f1e5a")
plt.legend(["Quantity", "Dollars"])
plt.ylabel("Freight Cost")
plt.show()

In [ ]:
vendor_df["Freight per unit"] = vendor_df["Freight"]/vendor_df["Quantity"]

In [ ]:
vendor_df

In [ ]:
low_quantity = vendor_df["Quantity"].quantile(0.25)
high_quantity = vendor_df["Quantity"].quantile(0.75)

In [ ]:
low_quantity

In [ ]:
high_quantity

In [ ]:
vendor_df.loc[vendor_df["Quantity"]<low_quantity, "Freight per unit"].mean()

In [ ]:
vendor_df.loc[vendor_df["Quantity"]>high_quantity, "Freight per unit"].mean()

In [ ]:
X = vendor_df[["Dollars"]]
y = vendor_df[["Freight"]]

In [ ]:
X.head()

In [ ]:
y.head()

In [ ]:
vendor_df.describe().round()

In [ ]:
# Train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_train

In [ ]:
linear_reg_model = LinearRegression()
linear_reg_model.fit(X_train, y_train)

In [ ]:
decision_tree_model = DecisionTreeRegressor(max_depth=4, random_state=42)
decision_tree_model.fit(X_train, y_train)

In [ ]:
random_forest_model = RandomForestRegressor(random_state=42)
random_forest_model.fit(X_train, y_train)

In [ ]:
# Evaluation function
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)*100
    
    print(f"{model_name} Performance: ")
    print(f"MSE: {mse}")
    print(f"MAE: {mae}")
    print(f"R2 score: {r2}")

In [ ]:
evaluate_model(linear_reg_model, X_test, y_test, "LinearRegression")

In [ ]:
evaluate_model(decision_tree_model, X_test, y_test, "DecisionTreeRegressor")

In [ ]:
evaluate_model(random_forest_model, X_test, y_test, "RandomForestRegressor")

In [ ]:
plt.scatter(X_test, y_test)
plt.plot(X_test, linear_reg_model.predict(X_test), color="red")

In [ ]:
input_data = {
    "Dollars" : [10303, 20303]
}

In [ ]:
df = pd.DataFrame(input_data)

In [ ]:
df

In [ ]:
linear_reg_model.predict(df)